In [0]:
#vamos para a camada gold. Camada gold é mais especializada, Até agora, deixamos esse dado viável para ser utilizado. Agora que o time da área de negócio correspondente entraria e falaria: "Esse é o cenário que eu preciso em específico". Então vamos ter alguns tratamentos em que teremos efetivamente regras de negócio. É olhar para o negócio e falar "Como respondo as minhas perguntas". Usamos para dashboards, analistas de dados e agentes.
#ferramentas de BI são direcionadas para o modelo dimensional (Kimball). Dentro desses modelos, temos o StarSchema e o Snowflake, por exemplo. StarSchema é o mais usado no mercado como modelagem de dados. StarSchema tem a estrutura de um cubo. Com uma tabela fato no centro e nas pontas temos as tabelas dimensões. Isso permite olhar o mesmo dado por perspectivas diferentes, por isso chama dimensões, dimensões de uma perspectiva a partir do mesmo fato
#Ex: pedido é um fato. Mas posso querer avaliar um mesmo pedido pela perspectiva de loja, de vendedores, de região, de clientes. São dimensões diferentes.
#Fato são dados quantitativos, mensuráveis = soma, cálculos, etc
#Dimensões são descritivas = pego aquele número e dou uma descrição pra ele. Ex: R$1000 vendidos por quem? Em qual região?
#Pros agentes, quanto mais joins na construção, maior chance de alucinação
#Sugestão  pra cenário agêntico: trabalhar com OBT (One big table) --> tipo de modelagem que foca em trabalhar os dados em uma única tabela. Une-se fatos e descrições na mesma tabela pra que ele consiga se encontrar melhor. Mas tudo depende do contexto, nem sempre OBT é a melhor opção. Se uma tabela fato tiver bem trabalhada e documentada, as vezes agente se sai bem com ela. Mas OBT é uma boa opção
#no mercado existem nomenclaturas diferentes para separar uma gold que é consumida por agentes e uma que é por dashboards, tem gente que chama de diamond, ou gold AI. Varia.
#DIMENSAO DEGENERADA: modelo dimensional para identificação do que saiu a partir de uma fato e isso gerou uma dimensão dentro dela pra poder acompanhar


In [0]:
%sql
--Vamos criar tabelas fato e dimensão e uma OBT
--Primeira coisa é a visão da dimensão. A ideia é dimensão de aeroporto, onde conseguimos descrever quais aeroportos (aerodromos) que temos a disposição

CREATE SCHEMA IF NOT EXISTS voebem.gold --criando o schema da gold

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- gold.dim_aeroporto — dimensao de aeroporto, servindo origem E destino.
--
-- A dimensao nasce do FATO, nao do cadastro. Copiar silver.aerodromos daria
-- 496 linhas e deixaria 218 aeroportos do fato orfaos (os estrangeiros, que a
-- ANAC nao cadastra). Uma dimensao existe para servir o fato: entao a lista de
-- chaves vem do fato, e o cadastro ENRIQUECE por LEFT JOIN.
--
-- Regra de negocio que nasce aqui: a classificacao pais_aeroporto pelo prefixo
-- ICAO. Isso nao podia estar na silver — e interpretacao, nao aritmetica.
-- ---------------------------------------------------------------------------
--vamos criar nossa tabela. vai vir da silver.aerodromos. vamos fazer um partition by com row number pra poder multiplicar as linhas sem dar nenhum erro --> localizador: #1
--Partition by divide os dados por uma chave, que no nosso caso, foi por icao. Por exemplo, todos os dados relativos ao ICAO de Guarulhos (tem um icao proprio) vao estar no mesmo arquivo. Vamos distribuir em varios arquivos usando uma chave. Se faz muito isso com data ou id. leitura de um arquivo só com tds dados seria muito dificultada.
--partition by é função de janela
CREATE OR REPLACE TABLE voebem.gold.dim_aeroporto AS
WITH aeroportos_do_fato AS ( --cte da fato para identificar quais aeroportos vao entrar na dimensao
    SELECT DISTINCT icao_origem AS icao FROM voebem.silver.vra 
    WHERE icao_origem IS NOT NULL AND icao_origem <> ''
    UNION
    SELECT DISTINCT icao_destino AS icao FROM voebem.silver.vra
    WHERE icao_destino IS NOT NULL AND icao_destino <> ''
),
--defesa: se a ANAC republicar o cadastro com ICAO repetido, o join multiplicaria  linhas do fato sem dar erro nenhum. Hoje são 496/496
cadastro AS ( -- cte de cadastro
    SELECT icao, nome, municipio, uf_nome, municipio servido, uf_servido_nome
    FROM(
        SELECT *, ROW_NUMBER() OVER (PARTITION BY icao ORDER BY nome) AS rn --#1
        FROM voebem.silver.aerodromos
        WHERE icao IS NOT NULL AND icao <> ''
    )
    WHERE rn = 1
)
--transformação da cte para a fato
SELECT
  a.icao                                                              AS icao_aeroporto,
  -- fallback textual obrigatorio: coluna que a IA vai ler nao pode vir NULL
  --tratamento de texto. concat é pra juntar dois textos que tão separados
  COALESCE(c.nome, concat('AEROPORTO FORA DO CADASTRO ANAC (', a.icao, ')')) 
                                                                      AS nome_aeroporto,
  c.municipio                                                         AS municipio_aeroporto,
  c.uf_nome                                                           AS uf_aeroporto,
  CASE WHEN a.icao RLIKE '^S[BDIJNSW]' THEN 'Brasil' ELSE 'Exterior' END
                                                                      AS pais_aeroporto,
  (c.icao IS NOT NULL)                                                AS no_cadastro_anac,
  current_timestamp()                                                 AS _processado_em
FROM aeroportos_do_fato a
LEFT JOIN cadastro c ON a.icao = c.icao

In [0]:
%sql
--Vamos conferir. ele vai plugar no nosso sql warehouse no databricks e fazer o select   
SELECT * FROM voebem.gold.dim_aeroporto 

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- gold.fato_voos — uma linha por etapa de voo.
--
-- E AQUI que nascem as regras de negocio que a silver nao podia ter:
--
--   1. PONTUALIDADE a 15 minutos (partida_pontual / chegada_pontual).
--      O numero 15 e decisao de cliente. Na silver ele fecharia porta;
--      aqui e uma linha de SQL que qualquer pessoa do negocio consegue ler.
--   2. ESCOPO domestico / internacional, a partir do tipo de linha.
--   3. As DECISOES SOBRE A QUARENTENA do marco-06 (documentadas abaixo).
--
-- Companhia e codigos de operacao entram como DIMENSAO DEGENERADA: codigo e
-- descricao no proprio fato, porque sao poucos atributos, nao mudam no tempo
-- e o consumidor final e um LLM — cada join a menos e um erro a menos.
--
-- ---------------------------------------------------------------------------
-- DECISOES SOBRE A QUARENTENA (213.543 registros diagnosticados no marco-06)
-- ---------------------------------------------------------------------------
-- a) AEROPORTO FORA DO CADASTRO DA ANAC (105.932) -> MANTIDO.
--    Nao e dado invalido, e aeroporto estrangeiro. Descartar mataria a
--    pergunta P4 do projeto. dim_aeroporto cobre 100% do fato.
-- b) VOO SEM HORARIO PREVISTO (30.800) -> MANTIDO.
--    O voo aconteceu e conta em "quantos voos". Sem horario previsto nao ha
--    atraso a calcular: a metrica fica NULL, e NULL ja se exclui sozinho de
--    qualquer media. Zerar seria mentir.
-- c) ATRASO FORA DA FAIXA PLAUSIVEL (778 partida / 823 chegada) -> LINHA
--    MANTIDA, METRICA ANULADA. Ha atrasos de ate 44.855 min (31 dias) e
--    antecipacoes de -43.057 (30 dias): erro de data na origem, nao operacao.
--    A linha continua contando como voo; a metrica vira NULL e a coluna
--    atraso_fora_de_faixa registra por que.
-- d) EMPRESA SEM CADASTRO (69) -> MANTIDA, com nome de fallback.
-- e) DUPLICATA EXATA (41) -> REMOVIDA. Unica exclusao de linha desta camada.
--    Duas linhas byte-identicas sao a mesma etapa publicada duas vezes;
--    conta-la duas vezes infla voos, cancelamentos e atraso ao mesmo tempo.
--    Grao esperado: 1.014.705 - 41 = 1.014.664.
-- ---------------------------------------------------------------------------
--Agora vamos criar nossa fato --> tabela que armazena o dado quantitativo --> numero e dados para calculo
--Aqui teremos infos dos voos, por exemplo. tempo de atraso, onde aconteceu 

CREATE OR REPLACE TABLE voebem.gold.fato_voos AS 
WITH vra_sem_duplicata AS (
    SELECT * FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY icao_empresa, numero_voo, codigo_di, 
                codigo_tipo_linha,
                                          icao_origem, icao_destino, partida_prevista, 
                                          partida_real, 
                                          chegada_prevista, chegada_real, situacao_voo
                ORDER BY _ingerido_em
            ) AS _rn    
            FROM voebem.silver.vra
    )
    WHERE _rn = 1
),
empresa AS ( -- cte pra construir empresas
    SELECT icao, razao_social, origem_cadastro
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY icao
            ORDER BY CASE WHEN situacao = 'ATIVA' THEN 0 ELSE 1 END, razao_social
        ) AS rn
        FROM voebem.silver.empresas
        WHERE icao IS NOT NULL AND icao <> ''
    )
    WHERE rn = 1
),
di AS ( --cte que pega codigo de operação
    SELECT codigo, descricao 
    FROM voebem.silver.codigos_operacao 
    WHERE dominio = 'codigo_di'
),
tipo_linha AS ( --cte para pegar tipo de linha que fica dentro da codigo_operacao
    SELECT codigo, descricao 
    FROM voebem.silver.codigos_operacao 
    WHERE dominio = 'codigo_tipo_linha'
),
base AS (--cte para identificação do que tá fora do que determinamos como padrão
    SELECT
        v.*,
        --decisao (c): metrica fora da faixa plausível vira NULL, linha fica
        (v.atraso_partida_min IS NOT NULL AND (v.atraso_partida_min < -120 OR v.atraso_partida_min > 1440))
            OR (v.atraso_chegada_min IS NOT NULL AND (v.atraso_chegada_min < -120 OR v.atraso_chegada_min > 1440))
                                                                                    AS atraso_fora_de_faixa
    FROM vra_sem_duplicata v
)
--select FINAL
SELECT
  -- ===== dimensao degenerada: companhia =====
  b.icao_empresa,
  COALESCE(e.razao_social, concat('COMPANHIA NAO CADASTRADA (', b.icao_empresa, ')'))
                                                                    AS nome_companhia,
  e.origem_cadastro                                                 AS cadastro_companhia,
  b.numero_voo,

  -- ===== dimensoes degeneradas: codigos de operacao =====
  b.codigo_di,
  COALESCE(d.descricao, concat('Codigo nao catalogado (', b.codigo_di, ')'))
                                                                    AS descricao_di,
  b.codigo_tipo_linha,
  COALESCE(t.descricao, concat('Codigo nao catalogado (', b.codigo_tipo_linha, ')'))
                                                                    AS descricao_tipo_linha,

  -- ===== REGRA DE NEGOCIO: escopo do voo =====
  CASE
    WHEN b.codigo_tipo_linha IN ('N', 'C') THEN 'Domestico'
    WHEN b.codigo_tipo_linha IN ('I', 'G') THEN 'Internacional'
    ELSE 'Nao classificado'
  END                                                               AS escopo_voo,

  -- ===== chaves para dim_aeroporto =====
  b.icao_origem,
  b.icao_destino,
  concat(b.icao_origem, ' - ', b.icao_destino)                      AS rota,

  -- ===== tempo =====
  b.partida_prevista,
  b.partida_prevista_data,
  b.partida_prevista_hora,
  hour(b.partida_prevista)                                          AS hora_partida_prevista,
  CASE dayofweek(b.partida_prevista_data)
    WHEN 1 THEN 'domingo'  WHEN 2 THEN 'segunda' WHEN 3 THEN 'terca'
    WHEN 4 THEN 'quarta'   WHEN 5 THEN 'quinta'  WHEN 6 THEN 'sexta'
    WHEN 7 THEN 'sabado'
  END                                                               AS dia_semana,
  date_trunc('MONTH', b.partida_prevista_data)                      AS mes_referencia,
  b.partida_real,
  b.chegada_prevista,
  b.chegada_real,

  -- ===== metricas (decisao (c) aplicada) =====
  CASE WHEN b.atraso_fora_de_faixa THEN NULL ELSE b.atraso_partida_min  END AS atraso_partida_min,
  CASE WHEN b.atraso_fora_de_faixa THEN NULL ELSE b.atraso_chegada_min  END AS atraso_chegada_min,
  CASE WHEN b.atraso_fora_de_faixa THEN NULL ELSE b.minutos_recuperados END AS minutos_recuperados,
  b.atraso_fora_de_faixa,

  -- ===== REGRA DE NEGOCIO: pontualidade a 15 minutos =====
  CASE WHEN b.atraso_fora_de_faixa OR b.atraso_partida_min IS NULL THEN NULL
       ELSE b.atraso_partida_min <= 15 END                          AS partida_pontual,
  CASE WHEN b.atraso_fora_de_faixa OR b.atraso_chegada_min IS NULL THEN NULL
       ELSE b.atraso_chegada_min <= 15 END                          AS chegada_pontual,

  -- ===== situacao =====
  b.situacao_voo,
  (b.situacao_voo = 'CANCELADO')                                    AS voo_cancelado,
  (b.situacao_voo = 'REALIZADO')                                    AS voo_realizado,

  current_timestamp()                                               AS _processado_em
FROM base b
LEFT JOIN empresa    e ON b.icao_empresa      = e.icao
LEFT JOIN di         d ON b.codigo_di         = d.codigo
LEFT JOIN tipo_linha t ON b.codigo_tipo_linha = t.codigo


In [0]:
#vamos criar nossa obt
#posso fazer obt baseada em views. ao inves de criar uma nova tabela gold, posso fazer uma view formatada em obt pra que eu tenha o msm dado que ta sendo gerado na gold alimentando minha obt. pra evitar que tenha manutenção em dois lugares falando do mesmo dado.
#quando olhamos para camada de obt pra um agente, faz sentido pensar numa view ao invés e criar uma tabela direto.

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- gold.obt_voos — One Big Table, desenhada para um consumidor especifico: uma IA.
--
-- Uma linha por etapa de voo, com TUDO resolvido: nome de companhia, nome de
-- aeroporto de origem e destino com municipio e UF, tipo de linha por extenso,
-- escopo, pontualidade e as metricas de atraso prontas.
--
-- Regra de ouro desta tabela: nenhuma coluna de codigo sem a coluna de
-- descricao correspondente ao lado. O LLM le nome, nao codigo ICAO.
--
-- E o unico join que ela exige de quem consome: nenhum.
-- ---------------------------------------------------------------------------
--centraliza todos dados numa mesma tabela ja linkando com a dimensao de aeroporto pra trazer tudo na mesma tabela de obt. ao invés de modelagem multitabelas, tenho uma unica com todas colunas lado a lado
CREATE OR REPLACE TABLE voebem.gold.obt_voos AS
SELECT
  -- ===== companhia =====
  f.icao_empresa,
  f.nome_companhia,
  f.numero_voo,

  -- ===== operacao =====
  f.codigo_di,
  f.descricao_di,
  f.codigo_tipo_linha,
  f.descricao_tipo_linha,
  f.escopo_voo,

  -- ===== origem =====
  f.icao_origem,
  o.nome_aeroporto        AS nome_aeroporto_origem,
  o.municipio_aeroporto   AS municipio_origem,
  o.uf_aeroporto          AS uf_origem,
  o.pais_aeroporto        AS pais_origem,

  -- ===== destino =====
  f.icao_destino,
  d.nome_aeroporto        AS nome_aeroporto_destino,
  d.municipio_aeroporto   AS municipio_destino,
  d.uf_aeroporto          AS uf_destino,
  d.pais_aeroporto        AS pais_destino,

  -- ===== rota, em codigo e por extenso =====
  f.rota                                                            AS rota_icao,
  concat(coalesce(o.municipio_aeroporto, f.icao_origem),  ' - ',
         coalesce(d.municipio_aeroporto, f.icao_destino))           AS rota_municipios,

  -- ===== tempo =====
  f.partida_prevista,
  f.partida_prevista_data,
  f.partida_prevista_hora,
  f.hora_partida_prevista,
  f.dia_semana,
  f.mes_referencia,
  f.partida_real,
  f.chegada_prevista,
  f.chegada_real,

  -- ===== metricas =====
  f.atraso_partida_min,
  f.atraso_chegada_min,
  f.minutos_recuperados,
  f.atraso_fora_de_faixa,
  f.partida_pontual,
  f.chegada_pontual,

  -- ===== situacao =====
  f.situacao_voo,
  f.voo_realizado,
  f.voo_cancelado,

  f._processado_em
FROM voebem.gold.fato_voos f --gerada a partir da fato voos. ao invés de construir algo que possa me gerar problema, constrói-se dados de algo já validado
LEFT JOIN voebem.gold.dim_aeroporto o ON f.icao_origem  = o.icao_aeroporto
LEFT JOIN voebem.gold.dim_aeroporto d ON f.icao_destino = d.icao_aeroporto

In [0]:
%sql
--verificação da tabela fato_voos
select * from voebem.gold.fato_voos

In [0]:
%sql
--verificação da tabela obt_voos
select * from voebem.gold.obt_voos

In [0]:
#agora, pra finalizar a camada gold, vamos olhar pra governança --> documentar, tagear e olhar pra lineage

In [0]:
#Governança da gold — documentação, tags e lineage
#Metadado, quando o consumidor é uma IA, deixa de ser documentação e vira requisito funcional. O COMMENT é literalmente o texto que o modelo lê para escolher qual coluna usar: coluna mal descrita é coluna usada errado, e o erro chega bonito, formatado e com número.

#Três entregas aqui:

# 1 -COMMENT em cada tabela e em cada coluna da obt_voos — descrevendo significado de negócio, não tipo de dado;

# 2 -tags nas tabelas gold;

# 3 - lineage do bronze até a OBT.

# - E um passo que não dá para pular: revisar as descrições geradas pela IA uma a uma, contra o dado. Uma delas está errada.
#---------------------------------------------------------------------------------------
#1. Revisão: a descrição que a IA errou
#Este foi o rascunho gerado para a métrica minutos_recuperados:

#"Minutos que a etapa recuperou em voo. Valor positivo indica que o voo chegou adiantado."

#Soa perfeito. E tá errado --> e o jeito de saber não é reler a frase, é perguntar para o dado.
########################################################################################


In [0]:
#VERIFICAÇÃO NO OBT DE INFOS DE TEMPO RECUPERADO, SE ATRASOU OU NAO
display(spark.sql("""
    SELECT
      COUNT(*)                                                      AS recuperou_algum_minuto,
      SUM(CASE WHEN atraso_chegada_min <= 0 THEN 1 ELSE 0 END)      AS chegou_adiantado_ou_no_horario,
      SUM(CASE WHEN atraso_chegada_min >  0 THEN 1 ELSE 0 END)      AS chegou_atrasado_mesmo_assim,
      SUM(CASE WHEN atraso_chegada_min > 15 THEN 1 ELSE 0 END)      AS chegou_atrasado_mais_de_15
    FROM voebem.gold.obt_voos
    WHERE minutos_recuperados > 0
"""))

In [0]:
#164.895 voos recuperaram tempo no ar e mesmo assim chegaram atrasados — 75.082 deles com mais de 15 minutos de atraso. A descrição da IA teria feito qualquer pessoa (e qualquer LLM) concluir o contrário.
#O certo é: positivo significa que a etapa chegou menos atrasada do que saiu. Não quer dizer que chegou no horário.

#A diferença entre as duas frases é a diferença entre "recuperou" e "resolveu". A segunda métrica, chegada_pontual, é quem responde se resolveu.

#Outras duas afirmações do rascunho que a revisão também derrubou:
# 1- RASCUNHO: "partida_pontual = false inclui os voos cancelados" --> O DADO DIZ: cancelado tem partida_pontual NULL: 29.140 de 29.140
# 2- RASCUNHO: "mes_referencia é o mês de todo voo" --> O DADO DIZ: é NULL em 30.798 voos, os que não têm horário previsto

In [0]:
#VERIFICAÇÃO DE SITUAÇÃO DE VOO, QUANTOS FORAM CANCELADOS E QUAIS DOS REALIZADOS TIVERAM PARTIDA PONTUAL
display(spark.sql("""
    SELECT situacao_voo,
           COUNT(*)                                                AS voos,
           SUM(CASE WHEN partida_pontual IS NULL THEN 1 ELSE 0 END) AS partida_pontual_null
    FROM voebem.gold.obt_voos GROUP BY situacao_voo
"""))

In [0]:
#PARTE DE METADADOS E GOVERNANÇA --> COMENTÁRIOS SOBRE NOSSAS TABELAS E SUAS COLUNAS PARA DOCUMENTAÇÃO

#A OBT é a tabela que a IA lê, mas o fato e a dimensão continuam sendo lidos por gente. Eles herdam as mesmas descrições onde a coluna é a mesma.

In [0]:
COMENTARIOS_OBT = {
    # ---- companhia ----
    "icao_empresa":           "Codigo ICAO de tres letras da companhia que operou a etapa. Use nome_companhia para exibir; este codigo serve para filtro exato.",
    "nome_companhia":         "Razao social da companhia aerea. Quando o codigo nao existe no cadastro da ANAC, traz COMPANHIA NAO CADASTRADA seguida do codigo, em vez de vazio.",
    "numero_voo":             "Numero comercial do voo divulgado pela companhia. Nao e identificador unico: o mesmo numero se repete todos os dias.",

    # ---- operacao ----
    "codigo_di":              "Codigo de autorizacao da etapa (DI) publicado pela ANAC. O codigo 1 aparece no dado e nao consta na tabela oficial de descricoes.",
    "descricao_di":           "Tipo da etapa por extenso: regular, extra, de retorno, charter, fretamento. Quando o codigo nao esta catalogado pela ANAC, diz isso explicitamente.",
    "codigo_tipo_linha":      "Codigo do tipo de linha da ANAC: N e C domesticas, I e G internacionais.",
    "descricao_tipo_linha":   "Tipo de linha por extenso, combinando escopo e natureza da operacao: Domestica Mista, Internacional Cargueira, etc.",
    "escopo_voo":             "Classificacao de negocio do voo em Domestico ou Internacional, derivada do tipo de linha. E a coluna certa para comparar os dois universos.",

    # ---- origem ----
    "icao_origem":            "Codigo ICAO do aeroporto de partida. Use nome_aeroporto_origem para exibir.",
    "nome_aeroporto_origem":  "Nome do aeroporto de partida. Aeroporto estrangeiro nao consta no cadastro da ANAC e aparece como AEROPORTO FORA DO CADASTRO ANAC seguido do codigo.",
    "municipio_origem":       "Municipio do aeroporto de partida. Vazio para aeroporto estrangeiro, que nao esta no cadastro brasileiro.",
    "uf_origem":              "Unidade federativa do aeroporto de partida, escrita POR EXTENSO (Sao Paulo, Ceara), como a ANAC publica. Nao e a sigla.",
    "pais_origem":            "Brasil ou Exterior, deduzido do prefixo do codigo ICAO. Serve para separar operacao domestica de internacional pelo lado do aeroporto.",

    # ---- destino ----
    "icao_destino":           "Codigo ICAO do aeroporto de chegada. Use nome_aeroporto_destino para exibir.",
    "nome_aeroporto_destino": "Nome do aeroporto de chegada. Mesma regra de fallback do aeroporto de origem.",
    "municipio_destino":      "Municipio do aeroporto de chegada. Vazio para aeroporto estrangeiro.",
    "uf_destino":             "Unidade federativa do aeroporto de chegada, por extenso.",
    "pais_destino":           "Brasil ou Exterior para o aeroporto de chegada.",

    # ---- rota ----
    "rota_icao":              "Rota no formato ORIGEM - DESTINO usando codigos ICAO. E a chave estavel para agrupar por rota.",
    "rota_municipios":        "Rota no formato municipio de origem - municipio de destino, para leitura humana. Aeroporto estrangeiro aparece pelo codigo ICAO, porque nao tem municipio no cadastro.",

    # ---- tempo ----
    "partida_prevista":       "Data e hora que a companhia programou para a partida, na hora local do aeroporto de origem.",
    "partida_prevista_data":  "Data programada da partida. Use para series diarias e para recortar periodo.",
    "partida_prevista_hora":  "Hora e minuto programados da partida, no formato HH:mm, para leitura.",
    "hora_partida_prevista":  "Hora cheia programada da partida, de 0 a 23. E a coluna certa para analisar o efeito cascata do atraso ao longo do dia.",
    "dia_semana":             "Dia da semana da partida programada, por extenso e em minusculas (domingo a sabado).",
    "mes_referencia":         "Primeiro dia do mes da partida programada, para agregacao mensal. E nulo nos voos que nao tem horario previsto informado.",
    "partida_real":           "Data e hora em que a aeronave efetivamente partiu. Nulo em voo cancelado.",
    "chegada_prevista":       "Data e hora programadas para a chegada, na hora local do aeroporto de destino.",
    "chegada_real":           "Data e hora em que a aeronave efetivamente pousou. Nulo em voo cancelado.",

    # ---- metricas ----
    "atraso_partida_min":     "Atraso de partida em minutos: horario real menos programado. Negativo significa que saiu adiantado. Nulo quando o voo foi cancelado, quando nao ha horario programado, ou quando o valor esta fora da faixa plausivel.",
    "atraso_chegada_min":     "Atraso de chegada em minutos: horario real menos programado. Negativo significa que pousou adiantado. Mesmas regras de nulo do atraso de partida.",
    "minutos_recuperados":    "Minutos que a etapa recuperou no ar: atraso de partida menos atraso de chegada. Positivo significa que chegou MENOS ATRASADA do que saiu, e NAO que chegou no horario - um voo pode recuperar 20 minutos e ainda assim pousar atrasado. Negativo significa que perdeu ainda mais tempo depois de decolar.",
    "atraso_fora_de_faixa":   "Verdadeiro quando o atraso calculado estava fora da faixa plausivel (menos de -2h ou mais de 24h), sinal de erro de data na origem. A linha continua contando como voo, mas as tres metricas de atraso foram anuladas.",
    "partida_pontual":        "Verdadeiro quando a partida atrasou 15 minutos ou menos, criterio de pontualidade deste projeto. Falso significa atraso maior que 15 minutos. Nulo significa que NAO DA para avaliar - voo cancelado ou sem horario programado - e nunca deve ser contado como atraso.",
    "chegada_pontual":        "Verdadeiro quando a chegada atrasou 15 minutos ou menos. Mesma regra de nulo da pontualidade de partida.",

    # ---- situacao ----
    "situacao_voo":           "Situacao informada pela companhia: REALIZADO quando a etapa aconteceu, CANCELADO quando nao aconteceu.",
    "voo_realizado":          "Verdadeiro quando a etapa foi realizada. Use como denominador de metricas operacionais.",
    "voo_cancelado":          "Verdadeiro quando a etapa foi cancelada. Voo cancelado NAO entra em nenhuma media de atraso, porque nao tem horario real; use esta coluna para taxa de cancelamento.",

    "_processado_em":         "Auditoria: momento em que esta linha foi construida na camada gold.",
}

for coluna, comentario in COMENTARIOS_OBT.items():
    spark.sql(f"ALTER TABLE voebem.gold.obt_voos ALTER COLUMN {coluna} COMMENT '{comentario}'")

print(f"{len(COMENTARIOS_OBT)} colunas comentadas em gold.obt_voos")


In [0]:
COMENTARIOS_DIM = {
    "icao_aeroporto":      "Codigo ICAO do aeroporto. Chave da dimensao, serve tanto para origem quanto para destino do fato.",
    "nome_aeroporto":      "Nome do aeroporto. Traz fallback textual com o codigo quando o aeroporto nao esta no cadastro da ANAC.",
    "municipio_aeroporto": "Municipio onde o aeroporto esta localizado. Vazio para aeroporto estrangeiro.",
    "uf_aeroporto":        "Unidade federativa por extenso, como a ANAC publica. Nao e a sigla.",
    "pais_aeroporto":      "Brasil ou Exterior, deduzido do prefixo ICAO. Regra de negocio criada na gold.",
    "no_cadastro_anac":    "Verdadeiro quando o aeroporto existe no cadastro de aerodromos publicos da ANAC. Falso e o esperado para aeroporto estrangeiro, e nao indica erro.",
    "_processado_em_":      "Auditoria: momento da construcao da dimensao.",
}

COMENTARIOS_FATO = dict(COMENTARIOS_OBT)
COMENTARIOS_FATO["icao_origem"]  = "Codigo ICAO do aeroporto de partida. Chave para gold.dim_aeroporto."
COMENTARIOS_FATO["icao_destino"] = "Codigo ICAO do aeroporto de chegada. Chave para gold.dim_aeroporto."
COMENTARIOS_FATO["rota"] = "Rota no formato ORIGEM - DESTINO usando codigos ICAO."
COMENTARIOS_FATO["cadastro_companhia"] = "De qual cadastro da ANAC veio a companhia: nacional ou estrangeira. Nulo quando o codigo nao tem cadastro."
COLUNAS_FATO = [c for c in COMENTARIOS_FATO if c not in (
    "nome_aeroporto_origem", "municipio_origem", "uf_origem", "pais_origem",
    "nome_aeroporto_destino", "municipio_destino", "uf_destino", "pais_destino",
    "rota_icao", "rota_municipios")]

for coluna in COLUNAS_FATO:
    spark.sql(f"ALTER TABLE voebem.gold.fato_voos ALTER COLUMN {coluna} COMMENT '{COMENTARIOS_FATO[coluna]}'")
print(f"{len(COLUNAS_FATO)} colunas comentadas em gold.fato_voos")

for coluna, comentario in COMENTARIOS_DIM.items():
    spark.sql(f"ALTER TABLE voebem.gold.dim_aeroporto ALTER COLUMN {coluna} COMMENT '{comentario}'")
print(f"{len(COMENTARIOS_DIM)} colunas comentadas em gold.dim_aeroporto")

In [0]:
#TAGEAMENTO --> COLOCANDO COMMENTS EM TABELAS E COLOCANDO TAGS

In [0]:
TABELAS_GOLD = {
    "voebem.gold.obt_voos": (
        "Gold - One Big Table de voos da ANAC, desnormalizada e desenhada para consumo por agente de IA. "
        "Uma linha por etapa de voo, com nomes ja resolvidos e metricas prontas: responde as perguntas de "
        "negocio do projeto sem nenhum JOIN. Criterio de pontualidade: 15 minutos. "
        "Voo cancelado nao tem metrica de atraso.",
        {"camada": "gold", "dominio": "aviacao", "consumo": "genie", "grao": "etapa_de_voo", "padrao": "obt"}, #aqui estamos falando que o objetivo de consumo principal da obt é o genie. Essa tabela é construída pensando no consumo agêntico, como posto anteriormente 
    ),
    "voebem.gold.fato_voos": (
        "Gold - fato de voos no grao de uma linha por etapa, com companhia e codigos de operacao como "
        "dimensoes degeneradas. E aqui que nascem as regras de negocio: pontualidade a 15 minutos, "
        "escopo domestico/internacional e as decisoes sobre a quarentena. "
        "Contagem = silver.vra menos 41 duplicatas exatas.",
        {"camada": "gold", "dominio": "aviacao", "consumo": "bi", "grao": "etapa_de_voo", "padrao": "fato"},
    ),
    "voebem.gold.dim_aeroporto": (
        "Gold - dimensao de aeroporto, servindo origem e destino do fato. Construida a partir dos codigos "
        "presentes no fato e enriquecida pelo cadastro da ANAC, para cobrir 100 por cento do fato inclusive "
        "os aeroportos estrangeiros, que a ANAC nao cadastra.",
        {"camada": "gold", "dominio": "aviacao", "consumo": "bi", "grao": "aeroporto", "padrao": "dimensao"}, #Aqui estamos declarando que essa coluna é pra consumo do BI
    ),
}

for tabela, (comentario, tags) in TABELAS_GOLD.items():
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    pares = ", ".join(f"'{k}' = '{v}'" for k, v in tags.items())
    spark.sql(f"ALTER TABLE {tabela} SET TAGS ({pares})") #alteração pra por as tags
    print(f"{tabela}: comentario + {len(tags)} tags")

In [0]:
#esses comentários são para gerar mais contexto e preparar nossos dados para serem consultados por agentes de IA, e podermos usar LLMs de maneira geral nessa consulta, nessa analise de dados. Isso tudo vai parar no sistema de metadados das tabelas que estamos construindo

In [0]:
#Pra encerrar nossa camada, vamos fazer a auditoria e verificar se realmente documentamos como o esperado, consultando nosso information_schema e ver se todas as tabelas estão devidamente ajustadas
display(spark.sql("""
    SELECT table_schema, table_name,
           COUNT(*)                                                         AS colunas,
           SUM(CASE WHEN comment IS NULL OR comment = '' THEN 1 ELSE 0 END) AS sem_comentario
    FROM voebem.information_schema.columns
    WHERE table_schema IN ('silver', 'gold')
    GROUP BY table_schema, table_name
    ORDER BY table_schema, table_name
"""))

In [0]:
display(spark.sql("""
    SELECT table_name, tag_name, tag_value
    FROM voebem.information_schema.table_tags
    WHERE schema_name = 'gold'
    ORDER BY table_name, tag_name
"""))

In [0]:
#Agora vamos entrar na parte do lineage
#No contexto de governança, a ideia do lineage é sabermos de onde surgiu o dado e como ele foi navegando pelo ambiente
#databricks traz o lineage de forma nativa com o unity catalog (parte de catalog --> só selecionar schemas e tabelas, e ver os metadados e lineage)
#lineage permite ter uma rastreabilidade absurda (de onde cada coluna e cada tabela vieram), e te permite pular uma etapa enorme de rastreabilidade de problemas
#conseguimos acessar isso por interface, mas também por via de código, igual abaixo:
display(spark.sql("""
    SELECT
      COALESCE(nullif(source_table_full_name, ''), '(arquivo no volume)') AS origem,
      target_table_full_name                                             AS destino
    FROM system.access.table_lineage --olhando dentro do system tables do databricks
    WHERE target_table_full_name LIKE 'voebem.%' --like ajuda buscar tudo que começa com esse prefixo, que comece com "voebem." e termine com qq coisa dps
      AND event_date >= current_date() - 7
    GROUP BY 1, 2
    ORDER BY destino, origem
"""))

In [0]:
#É esse grafo que responde, em dez segundos, a pergunta mais cara de um time de dados: "se eu mexer aqui, o que quebra lá na frente?".

#E ele responde também a pergunta do compliance, na direção contrária: "esse número no relatório veio de onde?" — obt_voos ← fato_voos ← silver.vra ← bronze.vra ← arquivo CSV da ANAC no volume, com data e hora de cada passo.